# Pretty Names

rename trees so that that have easier to use names for humans.

Steps:
- assign each cell a 4 chars genus name
- keep the original tree suffix
- rename the files

In [ ]:
#| default_exp pretty_names

In [ ]:
#| export
import geopandas as gpd
import pandas as pd
from pyprojroot import here
import string
from hilbertcurve.hilbertcurve import HilbertCurve
from pathlib import Path
from shapely.geometry import box
import shutil

In [ ]:
df = gpd.read_file(here("assets/the_crater_crowns.gpkg"))
df.head(3)

,GIS_ID,ID_number,area,area.1,neighbors,neighbors_truncated,geometry
0,CRA01-01_338470_8073610_10,338470_8073610_10,32.849655,32.849655,"338470_8073610_20,338470_8073620_04,338470_807...",False,"POLYGON ((338474.776 8073620.6, 338474.781 807..."
1,CRA01-01_338470_8073610_20_1,338470_8073610_20,18.991232,18.991232,"338470_8073610_10,338470_8073620_08,338470_807...",False,"POLYGON ((338479.11 8073616.981, 338479.064 80..."
2,CRA01-01_338470_8073620_01_1,338470_8073620_01,7.219352,7.219352,"338470_8073620_05,338470_8073630_05,338470_807...",False,"POLYGON ((338472.7 8073626.952, 338472.697 807..."


extract cell info (2D coordinates and id) from the cell name

In [ ]:
#|export
def extract_cell_info(df):
    extract_cell_id_re = r'(\d{6}_\d{7})'
    df['cell_id'] = df.GIS_ID.str.extract(extract_cell_id_re).squeeze()
    df['grid_x'] = df['cell_id'].str.split('_').str[0].astype(int)
    df['grid_y'] = df['cell_id'].str.split('_').str[1].astype(int)
    return df

In [ ]:
df = extract_cell_info(df)
len(df['cell_id'].unique())

48

## Get genus names


load the files the 4 chars genus names

In [ ]:
#| export
names = pd.read_csv(here("assets/short_genus_names.csv"))
names["name_lower"] = names["unit_name1"].str.lower()


In [ ]:

names

,Unnamed: 0,unit_name1,name_length,name_lower
0,0,Aata,4,aata
1,1,Abax,4,abax
2,2,Abra,4,abra
3,3,Acar,4,acar
4,4,Acca,4,acca
...,...,...,...,...
460,460,Zeus,4,zeus
461,461,Zoma,4,zoma
462,462,Zopa,4,zopa
463,463,Zora,4,zora



we have dumb algorithm to take `n` genus names in a attempt to maximise their diversity

TODO: probably replace with just sampling at random, is simpler and should work just as well

In [ ]:
#| export
def get_nth_genus(n) -> str | None:
    char_idx = n // 25
    letter_idx = n % 25
    sort_ascending = not bool(char_idx % 2) # to increase variability sort the 1st char in ascending order, the second in desceding and so on
    letter = string.ascii_lowercase[letter_idx]
    match = names.query(f"name_lower.str[{char_idx}] == '{letter}'")
    maybe_item =  match.sort_values('name_lower', ascending=sort_ascending).head(1).unit_name1
    return maybe_item.item() if len(maybe_item) > 0 else None

In [ ]:
get_nth_genus(50)

'Abax'

In [ ]:
#| export
def get_n_genera(n: int) -> set[str]:
    selected = set()
    i = 0
    while len(selected) < n:
        item = get_nth_genus(i)
        if item is not None: selected.add(item)
        i += 1
    return selected

In [ ]:
selected_genera = get_n_genera(48)
selected_genera

{'Aata',
 'Abax',
 'Bako',
 'Bubo',
 'Ceto',
 'Daku',
 'Ecto',
 'Faba',
 'Gaga',
 'Hada',
 'Iago',
 'Jaya',
 'Kali',
 'Lama',
 'Mada',
 'Naia',
 'Obus',
 'Odax',
 'Okea',
 'Omus',
 'Owra',
 'Oxya',
 'Paha',
 'Psoa',
 'Ptyx',
 'Raja',
 'Saba',
 'Spio',
 'Taia',
 'Thor',
 'Tyto',
 'Uaru',
 'Ubis',
 'Ucla',
 'Ugni',
 'Ulva',
 'Unio',
 'Urva',
 'Uvik',
 'Vaga',
 'Vini',
 'Weda',
 'Xana',
 'Yoda',
 'Zaus',
 'Zeus',
 'Zora',
 'Zuma'}

## Space filling curve

we want neighbouring cells to have alphabetically close genus names, so we use a spacefilling curve to map 2D cell indices to 1D indices.

parameters for an hilbert curve that empirically work decently

In [ ]:
#| export
def add_hilbert_index(df) -> pd.DataFrame:
    df['grid_x_norm'] = df['grid_x'] - df['grid_x'].min()
    df['grid_y_norm'] = df['grid_y'] - df['grid_y'].min()
    hilbert = HilbertCurve(8, 2)
    df['hilbert_dist'] = df.apply(lambda row: hilbert.distance_from_point([row['grid_x_norm'], row['grid_y_norm']]), axis=1)
    df.drop(columns=["grid_x_norm", "grid_y_norm"])
    return df

In [ ]:
add_hilbert_index(df)
df.head(3)

,GIS_ID,ID_number,area,area.1,neighbors,neighbors_truncated,geometry,cell_id,grid_x,grid_y,grid_x_norm,grid_y_norm,hilbert_dist
0,CRA01-01_338470_8073610_10,338470_8073610_10,32.849655,32.849655,"338470_8073610_20,338470_8073620_04,338470_807...",False,"POLYGON ((338474.776 8073620.6, 338474.781 807...",338470_8073610,338470,8073610,0,0,0
1,CRA01-01_338470_8073610_20_1,338470_8073610_20,18.991232,18.991232,"338470_8073610_10,338470_8073620_08,338470_807...",False,"POLYGON ((338479.11 8073616.981, 338479.064 80...",338470_8073610,338470,8073610,0,0,0
2,CRA01-01_338470_8073620_01_1,338470_8073620_01,7.219352,7.219352,"338470_8073620_05,338470_8073630_05,338470_807...",False,"POLYGON ((338472.7 8073626.952, 338472.697 807...",338470_8073620,338470,8073620,0,10,78


## Map cells to genus names 

In [ ]:
#| export
def get_cell_to_genus(df, selected_genera) -> dict[str, str]:
    cell_order = df.groupby('cell_id')['hilbert_dist'].first().sort_values()
    return dict(zip(cell_order.index, sorted(selected_genera)))

In [ ]:
cell_to_genus = get_cell_to_genus(df, selected_genera)

In [ ]:
#| export
def rename_trees(df, cell_to_genus: dict[str, str]) -> pd.DataFrame:
    df['suffix']  = df['GIS_ID'].str.extract(r'\d{6}_\d{7}_(.+)')
    df['pretty_name'] = df['cell_id'].map(cell_to_genus) + '_' + df['suffix']
    return df

In [ ]:
df = rename_trees(df, cell_to_genus)
df[['pretty_name', 'GIS_ID']].head()

,pretty_name,GIS_ID
0,Aata_10,CRA01-01_338470_8073610_10
1,Aata_20_1,CRA01-01_338470_8073610_20_1
2,Abax_01_1,CRA01-01_338470_8073620_01_1
3,Abax_04,CRA01-01_338470_8073620_04
4,Abax_05,CRA01-01_338470_8073620_05


## Renamer

In [ ]:
#| export
def rename_files(df, start_folder: Path, target_folder: Path, dry_run=False):
    name_map = dict(zip(df['GIS_ID'], df['pretty_name']))
    las_files = list(start_folder.glob('*.las'))
    for f in las_files:
        stem = f.stem
        if stem in name_map:
            new_name = target_folder / f"{name_map[stem]}.las"
            print(f"{f.name} -> {new_name.name}")
            if not dry_run: shutil.copy(f, new_name)
        else: print(f"Skipping {f.name} (not in map)")

## Cell grid

for convienence generate a cell grid shapefile with old and new names

In [ ]:
#| export
def make_grid(df, cell_to_genus, cell_size=10):
    grid = df[['cell_id', 'grid_x', 'grid_y']].drop_duplicates().copy()
    grid['pretty_name'] = grid['cell_id'].map(cell_to_genus)
    grid['geometry'] = grid.apply( lambda row: box(row['grid_x'], row['grid_y'], row['grid_x'] + cell_size, row['grid_y'] + cell_size),
            axis=1
        )
    grid = gpd.GeoDataFrame(grid, crs = df.crs)
    return grid

# Complete function

In [ ]:
#| export
def get_pretty_names(crowns_path: Path) -> tuple[pd.DataFrame, dict[str, str]]:
    df = gpd.read_file(crowns_path)
    df = extract_cell_info(df)
    selected_genera = get_n_genera(df['cell_id'].nunique())
    df = add_hilbert_index(df)
    cell_to_genus = get_cell_to_genus(df, selected_genera)
    df = rename_trees(df, cell_to_genus)
    return df, cell_to_genus